# 02 — Exploratory Data Analysis

**Purpose:** Understand the statistical properties and structural patterns of
the build graph and its associated metrics before modelling.  
Covers distribution plots, Pareto analysis, correlation analysis, degree
distributions, and dependency-adjacency heatmaps.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

# Load processed data
file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

# Load dependency graph
G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Distribution plots ────────────────────────────────────────────────
# Histograms for key numeric columns on log scale.

dist_cols = [c for c in ["compile_time_s", "sloc", "header_depth", "header_count",
                          "object_size_bytes", "link_time_s"]
             if c in target_metrics.columns]

n = len(dist_cols)
if n > 0:
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, col in enumerate(dist_cols):
        ax = axes[i]
        data = target_metrics[col].dropna()
        # Use log1p to handle zeros gracefully
        log_data = np.log1p(data)
        ax.hist(log_data, bins=60, edgecolor="black", alpha=0.75, color="steelblue")
        ax.set_xlabel(f"log(1 + {col})")
        ax.set_ylabel("count")
        ax.set_title(f"Distribution of {col}")
        ax.axvline(log_data.median(), color="red", linestyle="--", label=f"median={data.median():.2f}")
        ax.legend(fontsize=8)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Metric distributions (log scale)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No suitable numeric columns found for distribution plots.")

In [ ]:
# ── Pareto analysis (cumulative compile time vs target rank) ──────────
if "compile_time_s" in target_metrics.columns:
    ct = target_metrics["compile_time_s"].dropna().sort_values(ascending=False).reset_index(drop=True)
    cumulative = ct.cumsum() / ct.sum() * 100
    ranks = np.arange(1, len(ct) + 1)
    pct_ranks = ranks / len(ct) * 100

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar(pct_ranks, ct.values, width=pct_ranks[1] - pct_ranks[0], color="steelblue", alpha=0.6, label="compile time")
    ax1.set_xlabel("Target rank (% of total)")
    ax1.set_ylabel("Compile time (s)", color="steelblue")

    ax2 = ax1.twinx()
    ax2.plot(pct_ranks, cumulative.values, color="red", linewidth=2, label="cumulative %")
    ax2.set_ylabel("Cumulative compile time (%)", color="red")
    ax2.axhline(80, color="grey", linestyle="--", alpha=0.5)

    # Find the 80% mark
    idx_80 = np.searchsorted(cumulative.values, 80)
    pct_80 = pct_ranks[idx_80] if idx_80 < len(pct_ranks) else 100
    ax2.axvline(pct_80, color="grey", linestyle="--", alpha=0.5)
    ax1.set_title(f"Pareto: top {pct_80:.1f}% of targets account for 80% of compile time")

    fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
    plt.tight_layout()
    plt.show()
else:
    print("compile_time_s column not available.")

In [ ]:
# ── Correlation analysis (scatter plots, Pearson / Spearman) ──────────
from scipy import stats

corr_cols = [c for c in ["compile_time_s", "sloc", "header_depth", "header_count",
                          "object_size_bytes"]
             if c in target_metrics.columns]

if len(corr_cols) >= 2:
    # Correlation matrices
    pearson_corr = target_metrics[corr_cols].corr(method="pearson")
    spearman_corr = target_metrics[corr_cols].corr(method="spearman")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax1)
    ax1.set_title("Pearson correlation")
    sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax2)
    ax2.set_title("Spearman correlation")
    plt.tight_layout()
    plt.show()

    # Pairwise scatter plots for the most important pairs
    if "compile_time_s" in corr_cols:
        others = [c for c in corr_cols if c != "compile_time_s"]
        n_pairs = len(others)
        fig, axes = plt.subplots(1, n_pairs, figsize=(5 * n_pairs, 4))
        axes = np.atleast_1d(axes)
        for i, col in enumerate(others):
            ax = axes[i]
            valid = target_metrics[["compile_time_s", col]].dropna()
            ax.scatter(valid[col], valid["compile_time_s"], alpha=0.3, s=10)
            r_p, p_p = stats.pearsonr(valid[col], valid["compile_time_s"])
            r_s, p_s = stats.spearmanr(valid[col], valid["compile_time_s"])
            ax.set_xlabel(col)
            ax.set_ylabel("compile_time_s")
            ax.set_title(f"r_p={r_p:.2f}, r_s={r_s:.2f}")
        plt.suptitle("Compile time vs other metrics", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")

In [ ]:
# ── Dependency structure overview (degree distribution, DAG depth) ────
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# In-degree distribution
vals = list(in_deg.values())
axes[0].hist(vals, bins=range(0, max(vals) + 2), edgecolor="black", alpha=0.7, color="teal")
axes[0].set_xlabel("In-degree")
axes[0].set_ylabel("Count")
axes[0].set_title("In-degree distribution")
axes[0].set_yscale("log")

# Out-degree distribution
vals = list(out_deg.values())
axes[1].hist(vals, bins=range(0, max(vals) + 2), edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("Out-degree")
axes[1].set_ylabel("Count")
axes[1].set_title("Out-degree distribution")
axes[1].set_yscale("log")

# DAG depth per node (longest path from any root)
if nx.is_directed_acyclic_graph(G):
    topo_order = list(nx.topological_sort(G))
    depth = {n: 0 for n in G.nodes()}
    for node in topo_order:
        for succ in G.successors(node):
            depth[succ] = max(depth[succ], depth[node] + 1)
    max_depth = max(depth.values())
    depth_vals = list(depth.values())
    axes[2].hist(depth_vals, bins=range(0, max_depth + 2), edgecolor="black", alpha=0.7, color="mediumpurple")
    axes[2].set_xlabel("DAG depth")
    axes[2].set_ylabel("Count")
    axes[2].set_title(f"DAG depth distribution (max={max_depth})")
else:
    axes[2].text(0.5, 0.5, "Graph has cycles — not a DAG", ha="center", va="center",
                 transform=axes[2].transAxes, fontsize=12)
    axes[2].set_title("DAG depth (N/A)")

plt.tight_layout()
plt.show()

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(G)}")
if nx.is_directed_acyclic_graph(G):
    print(f"Max DAG depth: {max_depth}")

In [ ]:
# ── Heatmap of dependency adjacency matrix ───────────────────────────
# For large graphs we sample the top-N nodes by total degree to keep
# the heatmap readable.

MAX_NODES = 80
total_deg = {n: G.in_degree(n) + G.out_degree(n) for n in G.nodes()}
top_nodes = sorted(total_deg, key=total_deg.get, reverse=True)[:MAX_NODES]
sub = G.subgraph(top_nodes)

adj_matrix = nx.to_numpy_array(sub, nodelist=top_nodes)
labels = [str(n)[:20] for n in top_nodes]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(adj_matrix, xticklabels=labels, yticklabels=labels,
            cmap="YlOrRd", linewidths=0.1, ax=ax)
ax.set_title(f"Adjacency matrix (top {len(top_nodes)} nodes by degree)")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()